In [1]:
import pandas as pd
import numpy as np
import ollama
import re
import json

from src.io import read_tabular
from pathlib import Path

from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Literal

In [2]:
#Loading Data

fp = Path("../../data/test_semantic_extraction.csv")
df = read_tabular(fp)
input = df['sentence']
input = input.drop_duplicates(keep='first')

### Define Pydantic class:
Sentence - Subject - Verb - Object

Desired output: Series of JSON objects with the four vars, no grouping by sentence

In [3]:
class VerbSVO(BaseModel):
    sentence: str = Field(..., description="The full grammatical sentence you extracted the SVO from")
    subject: str = Field(..., description="The semantic subject related to the verb")
    verb: str = Field(..., description="The verb of the sentence or partial sentence")
    object: Optional[str] = Field(None, description="The semantic object related to the verb, if there is one")

class SentenceSVO(BaseModel):
    svos: List[VerbSVO] = Field(..., description="One entry per verb found in the sentence")

svo_schema = SentenceSVO.model_json_schema()

### Prompting

In [4]:
sysprompt_svo_zs = f'''
Your task is to read a sentence and extract semantic actors from them. For every verb in the sentence, extract the following four variables if possible:

- "sentence": The full grammatical sentence you were passed to analyze.
- "subject": The semantic subject associated with the action.
- "verb": The verb that indicates the action, in its infinitive form.
- "object": The semantic object associated with the verb, assuming there is one.

Observe the following rules:
- Subject and object should always be understood as semantic, rather than grammatical, categories. If there are indirect reports, passive sentences, etc. it is important that you parse the full sentence and identify the relevant semantic relations.
- All variables should be answered using words already in the text. If the subject or object is referenced using a pronoun, return that pronoun.
- If the subject is omitted from the sentence and can only be inferred from context, return "OMITTED" for the "subject" variable.
- Return the entire description of subject and object, including articles, potential adjectives, etc.
- If there are multiple objects phrased as a list separated by commas or connecting words like "and", create a new observation for each element of the list.

Output rules:
Return a JSON with a new entry for each verb in the sentence. For each verb you detect, return the variables according to the following JSON schema:
{svo_schema}

If there are multiple verbs or an object is separated into multiple observations as described above, make a new entry for each observation. Do not wrap in a code block. Do not return anything else.'''

### Setup

- Gemma-Cloud OR GPT-OSS 20b
- Temp = 0
- Seed = 42

### Setting options

In [5]:
modelname = "qwen3.8:latest"

opts = {
        "seed": 42,
        "temperature": 0.0
    }

output_file = "svo_qwen38.json"
log_file = "svo_qwen38_log.txt"

In [6]:
def model_exists(client, modelname):
    try:
        client.show(modelname)
        return True
    except ollama.ResponseError:
        return False

client = ollama.Client()

if model_exists(client, modelname):
    print(f"Model {modelname} found. Inference beginning.")
else:
    print(f"Model {modelname} not found. Pulling model.")
    ollama.pull(modelname)
    print("Model pulled successfully. Inference beginning.")
    
ctr=0
error_ctr=0
success_ctr=0
out = []
out_raw = []

for text in input:
    messages = [
            {"role": "system", "content": sysprompt_svo_zs},
            {"role": "user", "content": text}
        ]

    response = client.chat(
        model=modelname, 
        messages=messages, 
        options=opts,
        format=svo_schema,
        think=False)

    cleaned = response.message.content or "{}"

    ctr = ctr+1
    print(f"Processed {ctr} of {len(input)} sentences")
    out_raw.append(response)
    out.append(cleaned)

Model qwen3.8:latest found. Inference beginning.
Processed 1 of 30 sentences
Processed 2 of 30 sentences
Processed 3 of 30 sentences
Processed 4 of 30 sentences
Processed 5 of 30 sentences
Processed 6 of 30 sentences
Processed 7 of 30 sentences
Processed 8 of 30 sentences
Processed 9 of 30 sentences
Processed 10 of 30 sentences
Processed 11 of 30 sentences
Processed 12 of 30 sentences
Processed 13 of 30 sentences
Processed 14 of 30 sentences
Processed 15 of 30 sentences
Processed 16 of 30 sentences
Processed 17 of 30 sentences
Processed 18 of 30 sentences
Processed 19 of 30 sentences
Processed 20 of 30 sentences
Processed 21 of 30 sentences
Processed 22 of 30 sentences
Processed 23 of 30 sentences
Processed 24 of 30 sentences
Processed 25 of 30 sentences
Processed 26 of 30 sentences
Processed 27 of 30 sentences
Processed 28 of 30 sentences
Processed 29 of 30 sentences
Processed 30 of 30 sentences


## Saving results

Save once as raw txt (out), attempt validation and saving as JSON-validated block.

In [7]:
# Saving raw results for logging
with open(log_file, "w", encoding="utf-8") as f:
    f.write(str(out_raw))


In [8]:
json_out = []

for text in out:
    try:
        parsed = json.loads(text)
        validated = SentenceSVO(**parsed)
        json_out.append(validated.model_dump())

    except (ValidationError, json.JSONDecodeError) as e:
        print(f"Error: {e}")
        continue

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(json_out, f, indent=2, ensure_ascii=False)